# Task



More specifially, do the following:
1. A short EDA (Exploratory Data Analysis) of the housing data set.
2. Drop the column "ocean_proximity", then we have only have numeric columns which will simplify your analysis.
3. Split your data into train, validation and test set. Before this, split your data into y and X. 
4. You have missing values in your data (not sure you do but you can assume so). Handle this with [ SimpleImputer(strategy="median") ], check the fantastic Scikit-learn documentation for details. Notice, the SimpleImputer should only be used for transformation on the validation and test data. Not fitting. 
5. Create one "Linear Regression" model and one "Lasso" model. For the Lasso model, use GridSearchCV to optimize alpha values,

7. Which model is best on the validation data? 

8. Evaluate your chosen model on the test set using the root mean squared error (RMSE) as the metric.


# Code

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

: 

In [ ]:
# Below, set your own path where you have stored the data file. 
housing = pd.read_csv(r'C:housing.csv')

## EDA

In [ ]:
housing.head()

In [ ]:
housing.info()

## Data

In [ ]:
# Drop non-numeric column
data = housing.drop(columns=['ocean_proximity'])

In [ ]:
# Split data into features and target
X = data.drop(columns=['median_house_value'])
y = data['median_house_value']

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=36)

In [ ]:
# X_train.info()
# X_val.info()
# X_test.info()

In [ ]:
print(X_train.shape)
print(y_train.shape)

### Intermezzo - A plot that helps us understand the problem 

In [ ]:
import matplotlib.image as mpimg

train_full, test = train_test_split(housing, test_size=0.2, random_state=40)
train, val = train_test_split(train_full, test_size=0.25, random_state=36)

california_img=mpimg.imread('california.png') 

ax = train.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
s=train['population']/100, label="Population",
c="median_house_value", cmap=plt.get_cmap("jet"),
colorbar=False, alpha=0.4)
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,
cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)
prices = train["median_house_value"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
cbar = plt.colorbar(ticks=tick_values/prices.max())
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value', fontsize=16)
plt.legend(fontsize=16)

### Intermezzo ending --------------------

In [ ]:
my_imputer = SimpleImputer(strategy='median')

X_train = my_imputer.fit_transform(X_train)
X_val = my_imputer.transform(X_val)

X_train_full = my_imputer.fit_transform(X_train_full)   
X_test = my_imputer.transform(X_test)

In [ ]:
# Notice, after using SimpleImputer, the data is trasformed to NumPy array. We don't need to care now but good to know. 
# type(X_train_full)

## Modelling

In [ ]:
from sklearn.linear_model import LinearRegression  # Normally placed at the top
lin_reg = LinearRegression()

lin_reg.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import Lasso  # Normally placed at the top

lasso = Lasso()
hyperparam_grid = {'alpha': [0.01, 0.1, 1, 10, 50, 100]}
grid_search = GridSearchCV(lasso, hyperparam_grid, scoring='neg_root_mean_squared_error', cv=5)
grid_search.fit(X_train, y_train)

In [ ]:
# Analysis of the grid search
# pd.DataFrame(grid_search.cv_results_)

In [ ]:
lr_pred_val = lin_reg.predict(X_val)
lasso_pred_val = grid_search.predict(X_val)

print('RMSE Linear Regression:', root_mean_squared_error(y_val, lr_pred_val))
print('RMSE Lasso Regression:', root_mean_squared_error(y_val, lasso_pred_val))

## Evaluation of best model on test data

In [ ]:
final_model_lr = LinearRegression()

final_model_lr.fit(X_train_full, y_train_full)

In [ ]:
lr_pred_test = grid_search.predict(X_test)
rmse_test = root_mean_squared_error(y_test, lr_pred_test)
print('RMSE Linear Regression on Test data:', rmse_test)

In [ ]:
# We do a prediction error of around 30% relative to average value in t_test. Not bad?
rmse_test/np.mean(y_test)